# PHASE 5: FEATURE ENGINEERING & RAW → PROCESSED DATA PIPELINE


## 1. Business Objective
The objective of this phase is to establish a rigorous, reproducible analytical data pipeline (Raw $\rightarrow$ Processed). We will derive statistically justified predictor variables while enforcing a strict audit against **Target Leakage** to guarantee the integrity of future ML models.


## 2. Feature Engineering Questions
- Which Analytical Grain correctly isolates our statistical findings?
- How do we synthesize physical infrastructure capacities into algorithmic inputs?
- Which observed outcomes (like Wait Time or Sessions) constitute fatal Target Leakage if predicting Congestion?
- Which categorical profiles (Station Type, Charger Type) require formal ML encoding strategies?


## 3. Load Raw Data
Raw data remains completely immutable. We ingest validated primitives to construct our feature matrices.


In [1]:
import pandas as pd
import numpy as np

RAW_DIR = r"C:\PYTHON p45\EV-Charging-Infrastructure-Analytics-Demand-Forecasting-Utilization-Optimization\Project_files\data\raw"
PROCESSED_DIR = r"C:\PYTHON p45\EV-Charging-Infrastructure-Analytics-Demand-Forecasting-Utilization-Optimization\Project_files\data\processed"

stations = pd.read_csv(f"{RAW_DIR}/stations.csv")
metrics = pd.read_csv(f"{RAW_DIR}/station_hourly_metrics.csv")
sessions = pd.read_csv(f"{RAW_DIR}/charging_sessions.csv")


## 4. Define Analytical Grain
**Grain Selected:** **Station-Level Dataset (One row = One Charging Station)**.

**Justification:** 
Phases 3A and 4 definitively proved that temporal variance in this dataset is highly uniform, while spatial/structural variance is highly predictive. Time-series forecasting for this setup is not analytically defensible. Conversely, determining why specific architectural setups hit high congestion or yield is heavily supported by the statistical facts. We therefore aggregate all temporal metrics directly into structural cross-sections.


## 5. Data Preparation & 6. Station-Level Features
We integrate foundational static attributes (Predictors) and roll up historical observed outcomes (Targets/Risk Variables).


In [2]:
# 1. Base Predictors (Safe at Station Level)
df_features = stations[['Station_ID', 'Number_of_Chargers', 'Max_Station_Power_kW', 
                        'Parking_Spots', 'Station_Age_Years', 'Charger_Type', 
                        'Station_Type', 'Renewable_Energy_Source']].copy()

# 2. Derive observed aggregates
station_agg = metrics.groupby('Station_ID').agg(
    Avg_Utilization=('Utilization_Rate', 'mean'),
    Congestion_Freq=('Congestion_Flag', 'mean'),
    Total_Sessions=('Sessions_Count', 'sum'),
    Avg_Wait_Time=('Avg_Wait_Time_Min', 'mean')
).reset_index()

# Join
df_processed = pd.merge(df_features, station_agg, on='Station_ID', how='inner')


## 7. Session-Level Features
As we operate on the Station Grain, session-level fields are summarized into Station-Level averages.



In [3]:
# E.g., What is the average session duration at this specific location?
session_agg = sessions.groupby('Station_ID').agg(
    Avg_Session_Duration=('Charging_Duration_Min', 'mean')
).reset_index()

df_processed = pd.merge(df_processed, session_agg, on='Station_ID', how='left')


## 8. Capacity & Utilization Features
Engineered vectors combining inputs into density representations.

**`Power_per_Charger`** = Max Station Power / Number of Chargers $\rightarrow$ Represents actual max output threshold.
**`Target_High_Util`** = Boolean classifying if Avg_Utilization > 75th Percentile.


In [4]:
# Feature Engineering
df_processed['Power_per_Charger'] = df_processed['Max_Station_Power_kW'] / df_processed['Number_of_Chargers']
df_processed['Charger_to_Parking_Ratio'] = df_processed['Number_of_Chargers'] / df_processed['Parking_Spots']

# Target Classification
q75_util = df_processed['Avg_Utilization'].quantile(0.75)
df_processed['Target_High_Util'] = (df_processed['Avg_Utilization'] > q75_util).astype(int)


## 9. Congestion Features
Distinguishing Inputs vs Targets.
We define `Target_High_Congestion` if the `Congestion_Freq` is in the upper quartile.


In [5]:
q75_cong = df_processed['Congestion_Freq'].quantile(0.75)
df_processed['Target_High_Congestion'] = (df_processed['Congestion_Freq'] > q75_cong).astype(int)


## 10. Temporal Features
**Omitted.** The uniform distributions confirmed in Phase 3A render calendar/time metrics mathematically irrelevant as predictive features for station structural optimization.


## 11. Categorical Features
Categorical variables must be labeled. For now, we preserve readable states, but map their intended future encodings.


In [6]:
# Mapping intended strategies (to be executed in ML Pipeline Phase)
cat_fields = ['Charger_Type', 'Station_Type', 'Renewable_Energy_Source']
for col in cat_fields:
    print(f"Categorical Field '{col}' has unique values: {df_processed[col].unique()}")
    # ML Strategy: One-Hot Encoding during Phase 6 pipeline construction.


Categorical Field 'Charger_Type' has unique values: ['AC Level 2' 'DC Fast Charger' 'AC Level 1']
Categorical Field 'Station_Type' has unique values: ['Ultra-Fast Charging' 'Fast Charging' 'Destination/AC']
Categorical Field 'Renewable_Energy_Source' has unique values: ['Yes' 'No']


## 12. CRITICAL — DATA LEAKAGE AUDIT
If we intend to predict whether a blueprint will yield a highly congested or highly utilized station (before it is built), incorporating performance markers that only logically exist *after* it opens guarantees fatal Target Leakage.

| Feature | Audit Status | Reasoning |
| :--- | :--- | :--- |
| `Number_of_Chargers` | **SAFE** | Known at design time. |
| `Max_Station_Power_kW` | **SAFE** | Known at design time. |
| `Station_Type` | **SAFE** | Known at design time. |
| `Avg_Utilization` | **TARGET** | This is the outcome we intend to model. |
| `Total_Sessions` | **DEFINITE LEAKAGE** | Fully dependent on outcome. Cannot be an input. |
| `Avg_Wait_Time` | **DEFINITE LEAKAGE** | By-product of congestion itself. |
| `Congestion_Freq` | **TARGET** | Predictive outcome classification. |


## 13. Feature Validation
Programmatic verification of the engineered DataFrame.


In [7]:
validation_report = pd.DataFrame({
    'Missing_Values': df_processed.isnull().sum(),
    'Data_Type': df_processed.dtypes,
    'Min': df_processed.select_dtypes(include=[np.number]).min(),
    'Max': df_processed.select_dtypes(include=[np.number]).max()
})
display(validation_report.fillna('Categorical'))


,Missing_Values,Data_Type,Min,Max
Avg_Session_Duration,0,float64,44.781463,73.223333
Avg_Utilization,0,float64,0.096816,0.952287
Avg_Wait_Time,0,float64,0.256667,10.876957
Charger_Type,0,object,Categorical,Categorical
Charger_to_Parking_Ratio,0,float64,0.333333,1.0
Congestion_Freq,0,float64,0.0,0.91453
Max_Station_Power_kW,0,int64,22.0,2800.0
Number_of_Chargers,0,int64,1.0,8.0
Parking_Spots,0,int64,1.0,10.0
Power_per_Charger,0,float64,22.0,350.0


## 14. Save Processed Datasets
We export the verified Station-level DataFrame. We intentionally bypass creating Session-Level and Station-Hour derivatives to strictly enforce the analytical clarity derived in our phases.


In [8]:
target_path = f"{PROCESSED_DIR}/station_features.csv"
df_processed.to_csv(target_path, index=False)
print(f"Successfully saved {len(df_processed)} engineered rows to {target_path}")


Successfully saved 5000 engineered rows to C:\PYTHON p45\EV-Charging-Infrastructure-Analytics-Demand-Forecasting-Utilization-Optimization\Project_files\data\processed/station_features.csv


## 15. Feature Dictionary / Lineage

| Feature | Source | Transformation | Grain | ML Role | Leakage Status |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `Station_ID` | `stations.csv` | Direct | Station | Index | SAFE |
| `Number_of_Chargers` | `stations.csv` | Direct | Station | Predictor | SAFE |
| `Power_per_Charger` | Both | `Max_Kw / Chargers` | Station | Predictor | SAFE |
| `Target_High_Util` | `metrics.csv` | `Avg > 75th %` | Station | TARGET | TARGET |
| `Avg_Wait_Time` | `metrics.csv` | `mean()` | Station | Excluded | LEAKAGE |


## 16. Final ML Definitions
Based on the statistical relationships mapped and the robust safety audit above, we specify the formal ML problems.

### Problem B: Station Congestion Risk
*Predict whether a specific station blueprint operates under high congestion.*

**Predictors (Core Features):** 
`Number_of_Chargers`, `Max_Station_Power_kW`, `Parking_Spots`, `Station_Age_Years`, `Charger_Type`, `Station_Type`, `Renewable_Energy_Source`, `Power_per_Charger`.

**Target:**
`Target_High_Congestion` (Binary Classification).

**EXCLUDED (Leakage Hazard):**
`Total_Sessions`, `Avg_Utilization`, `Avg_Wait_Time`, `Avg_Session_Duration`.


## 17. Phase 5 Conclusion & 18. Next Phase 
1. **Processed Datasets:** `data/processed/station_features.csv` has been built and committed.
2. **Grain:** 1 Row = 1 Station (Aggregated Baseline).
3. **Engineered Features:** Derived load balances (`Power_per_Charger`, `Target_High_Util`).
4. **Strongest Features:** Infrastructure variables (Power, Chargers, Layout types).
5. **Leakage Risks:** Excluded all downstream performance indicators (Wait Times, Absolute Session counts) from input arrays.
6. **Most Defensible ML Problem:** Predicting High Congestion or High Utilization *Classification* based fundamentally on physical specs.

**Readiness:** The data architecture is clean. We are ready to initialize **Phase 6: Machine Learning Pipeline**.
